# Belief explorer

One sequence at a time, with every parameter live: the physical system and its
constants, the chain, the observation channel, and which transformer checkpoint
reads it. Both trained architectures load from the same dropdown - the shape is
recovered from the weights, so `d128l4` and `d256l8` need no separate handling.

**What is fitted and what is not.** The bottom panel is the model's own output and
has nothing between it and the picture. The belief and metric panels go through a
ridge probe fitted here, on 768 sequences drawn with a different seed from the one
on screen. That makes them responsive rather than authoritative - for a number to
quote, use `experiments/06_reference_probe.py`.

**Changing physics under a fixed checkpoint is the interesting move, not a bug.**
The model was trained on the defaults in `physics/messk_configs.py`. Push `delta_v`
or `dt` away from those and you are asking what the learned representation does
off its training distribution, which the next-token panel will show going soft
well before the belief panel does.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import ipywidgets as W
import numpy as np
from IPython.display import display

from models.train import pick_device
from models.explore import find_checkpoints, fit_readout, load_model, plot_sequence
from physics.messk_configs import CHAIN, DRIVER, MESSK_CONFIGS, SYSTEMS, make_process

OUTPUT_DIR = Path("../experiments/outputs-messk")
DEVICE = pick_device()  # mps on this Mac; the 8-layer model is slow on cpu

checkpoints = find_checkpoints(OUTPUT_DIR)
print(f"{len(checkpoints)} checkpoints under {OUTPUT_DIR.resolve()}")
if not checkpoints:
    print("none found - copy them over, e.g.\n"
          "  scp -r staging-entity:~/belief-in-physics/experiments/outputs-messk/'*.pt' "
          f"{OUTPUT_DIR}/")
print(f"device {DEVICE}")


## Controls

`m` and `n_steps` change the sequence length, and the checkpoint's position
embedding is sized for the length it was trained on, so raising them past the
default is refused rather than silently truncated. Lowering them is fine.

Defaults for every constant come from `physics/messk_configs.py`, so the reset
button is just a reload of that file.

In [ ]:
def _system_of(name):
    return MESSK_CONFIGS[name]

config_w   = W.Dropdown(options=sorted(MESSK_CONFIGS), value="pendulum_mess4", description="system")
ckpt_w     = W.Dropdown(options=sorted(checkpoints) or ["<none found>"], description="checkpoint")
delta_v_w  = W.FloatText(value=SYSTEMS["pendulum"]["delta_v"], step=0.05, description="delta_v")
dt_w       = W.FloatText(value=SYSTEMS["pendulum"]["dt"], step=0.005, description="dt")
n_steps_w  = W.IntSlider(value=DRIVER["n_steps"], min=2, max=20, description="n_steps")
m_w        = W.IntSlider(value=DRIVER["m"], min=4, max=32, description="ticks (m)")
n_obs_w    = W.IntSlider(value=DRIVER["n_obs"], min=11, max=361, step=10, description="n_obs")
stay_w     = W.FloatSlider(value=CHAIN["stay"], min=0.25, max=0.99, step=0.01, description="stay")
alpha_w    = W.FloatSlider(value=CHAIN["alpha"], min=0.25, max=0.99, step=0.01, description="alpha")
seq_w      = W.IntSlider(value=0, min=0, max=63, description="sequence")
seed_w     = W.IntText(value=20260829, description="data seed")
depths_w   = W.IntSlider(value=4, min=1, max=9, description="probe depths")
whole_w    = W.Checkbox(value=True, description="whole-tick readout")
ridge_w    = W.FloatLogSlider(value=100.0, base=10, min=-1, max=5, description="ridge alpha")
nfit_w     = W.IntSlider(value=768, min=256, max=3072, step=256, description="fit seqs")

def _sync_system_defaults(change):
    spec = SYSTEMS[_system_of(change["new"])]
    delta_v_w.value, dt_w.value = spec["delta_v"], spec["dt"]

config_w.observe(_sync_system_defaults, names="value")

panel = W.HBox([
    W.VBox([config_w, ckpt_w, seq_w, seed_w]),
    W.VBox([delta_v_w, dt_w, n_steps_w, m_w, n_obs_w]),
    W.VBox([stay_w, alpha_w, depths_w, whole_w, ridge_w, nfit_w]),
])

## The view

Each redraw refits the probe, so it takes a second or two. The fit is cached on
the parameter tuple, so moving only the sequence slider is instant.

In [ ]:
_cache = {}

def _build(config, ckpt, delta_v, dt, n_steps, m, n_obs, stay, alpha, depths, whole, ridge, n_fit, device=None):
    device = device or DEVICE
    key = (config, ckpt, delta_v, dt, n_steps, m, n_obs, stay, alpha, depths, whole, ridge, n_fit)
    if key in _cache:
        return _cache[key]
    proc = make_process(
        config, delta_v=delta_v, dt=dt, n_steps=n_steps, m=m, n_obs=n_obs,
        chain={"stay": stay, "alpha": alpha},
    )
    model = load_model(checkpoints[ckpt], device=device)
    if proc.seq_len > model.config.n_ctx:
        raise ValueError(
            f"sequence is {proc.seq_len} tokens but this checkpoint's context is "
            f"{model.config.n_ctx}; lower m or n_steps"
        )
    if proc.n_obs > model.config.vocab_size:
        raise ValueError(
            f"n_obs {proc.n_obs} exceeds the checkpoint vocabulary {model.config.vocab_size}"
        )
    readout = fit_readout(
        model, proc, n_fit=n_fit,
        n_depths=min(depths, model.config.n_layers + 1),
        whole_tick=whole, alpha=ridge,
    )
    _cache[key] = (proc, model, readout)
    return _cache[key]


def draw(config, ckpt, delta_v, dt, n_steps, m, n_obs, stay, alpha, sequence, seed, depths, whole, ridge, n_fit):
    if ckpt not in checkpoints:
        print("no checkpoint available")
        return
    proc, model, readout = _build(config, ckpt, delta_v, dt, n_steps, m, n_obs, stay, alpha, depths, whole, ridge, n_fit)
    batch = proc.sample_batch(np.random.default_rng(seed), sequence + 1)
    print(f"{model.config.n_layers} layers, d_model {model.config.d_model}, "
          f"{model.config.n_heads} heads | seq_len {proc.seq_len} | "
          f"probe depths {readout.depths} dim {readout.ridge.coef_.shape[1]}")
    plot_sequence(proc, batch, sequence, model, readout, DEVICE)


out = W.interactive_output(draw, dict(
    config=config_w, ckpt=ckpt_w, delta_v=delta_v_w, dt=dt_w, n_steps=n_steps_w,
    m=m_w, n_obs=n_obs_w, stay=stay_w, alpha=alpha_w, sequence=seq_w, seed=seed_w,
    depths=depths_w, whole=whole_w, ridge=ridge_w, n_fit=nfit_w,
))
display(panel, out)

## Things worth trying

- **Same system, checkpoint 0 against the final one.** `*_0.pt` is the random
  initialisation. The metric panel barely moves; the belief panel is where training
  shows up. That contrast is the central result of the experiment.
- **`d128l4` against `d256l8` at the same system.** Belief improves, metric does
  not, because the metric is nearly a finite difference of the tokens on screen.
- **Turn `whole-tick` off.** The readout drops to the tick's last position only and
  the metric panel collapses, which is the readout effect isolated from the model.
- **Raise `stay` towards 0.99.** The chain becomes sticky, the belief flattens onto
  one corner, and a high R2 stops meaning anything - worth seeing once so the
  entropy caveat in the results is concrete.
- **Walk `delta_v` down towards zero.** The letters stop being distinguishable from
  the observations, and the belief readout degrades towards the prior.